# 05 - Explorative Datenanalyse: Oelpreise (Yahoo Finance)

**Ziel:** Oelpreisdaten in USD von Yahoo Finance laden, erkunden und erste Qualitaetspruefung durchfuehren.

**Rohstoffe:** WTI Crude Oil, Brent Crude Oil

**Datenquelle:** Yahoo Finance (via yfinance Library)

---

## 1. Setup und Imports

In [ ]:
# Bibliotheken importieren
import yfinance as yf          # Yahoo Finance API
import pandas as pd            # Datenverarbeitung
import numpy as np             # Numerische Berechnungen
import matplotlib.pyplot as plt # Visualisierung
import seaborn as sns           # Erweiterte Visualisierung
import os                       # Dateipfade

# Darstellung konfigurieren
plt.style.use('seaborn-v0_8')  # Schoenerer Plot-Stil
plt.rcParams['figure.figsize'] = (12, 6)  # Standardgroesse fuer Plots
pd.set_option('display.max_columns', None)  # Alle Spalten anzeigen

print('Setup erfolgreich!')

## 2. Daten laden von Yahoo Finance

Wir laden die Oelpreisdaten fuer WTI und Brent Crude Oil.
Yahoo Finance verwendet `CL=F` fuer WTI Crude Oil Futures und `BZ=F` fuer Brent Crude Oil Futures. Preise sind in USD pro Barrel.

In [ ]:
# Konfiguration: Welche Rohstoffe und welcher Zeitraum?
OIL_TICKERS = {
    'CL=F': 'WTI_Crude_Oil',    # WTI Crude Oil Futures (USD/Barrel)
    'BZ=F': 'Brent_Crude_Oil',  # Brent Crude Oil Futures (USD/Barrel)
}

START_DATE = '2022-01-01'
END_DATE = '2026-03-25'

print(f'Zeitraum: {START_DATE} bis {END_DATE}')
print(f'Rohstoffe: {list(OIL_TICKERS.values())}')

In [ ]:
# Daten laden und in einem Dictionary speichern
oil_data = {}

for yahoo_symbol, oil_name in OIL_TICKERS.items():
    print(f'Lade {oil_name} ({yahoo_symbol})...')
    
    # Daten herunterladen
    ticker = yf.Ticker(yahoo_symbol)
    df = ticker.history(start=START_DATE, end=END_DATE)
    
    # Im Dictionary speichern
    oil_data[oil_name] = df
    
    print(f'  -> {len(df)} Zeilen geladen')

print('\nAlle Daten geladen!')

## 3. Erste Datenuebersicht

Schauen wir uns an, was wir bekommen haben.

In [ ]:
# Uebersicht fuer jeden Rohstoff
for oil_name, df in oil_data.items():
    print(f'\n{"=" * 50}')
    print(f'{oil_name}')
    print(f'{"=" * 50}')
    print(f'Shape (Zeilen x Spalten): {df.shape}')
    print(f'Zeitraum: {df.index.min()} bis {df.index.max()}')
    print(f'Spalten: {list(df.columns)}')
    print(f'Datentypen:\n{df.dtypes}')
    print(f'\nErste 3 Zeilen:')
    display(df.head(3))

## 4. Datenqualitaetspruefung

Wir pruefen:
- Fehlende Werte (NaN)
- Duplikate
- Datentypen
- Statistische Kennzahlen

In [ ]:
# Fehlende Werte pruefen
print('FEHLENDE WERTE PRO ROHSTOFF')
print('=' * 50)

for oil_name, df in oil_data.items():
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    
    print(f'\n{oil_name}:')
    if missing.sum() == 0:
        print('  Keine fehlenden Werte!')
    else:
        for col in df.columns:
            if missing[col] > 0:
                print(f'  {col}: {missing[col]} fehlend ({missing_pct[col]}%)')

In [ ]:
# Duplikate pruefen (gleiche Datumswerte)
print('DUPLIKATE PRO ROHSTOFF')
print('=' * 50)

for oil_name, df in oil_data.items():
    dupes = df.index.duplicated().sum()
    print(f'{oil_name}: {dupes} Duplikate')

In [ ]:
# Statistische Kennzahlen (Deskriptive Statistik)
for oil_name, df in oil_data.items():
    print(f'\n{"=" * 50}')
    print(f'Deskriptive Statistik: {oil_name}')
    print(f'{"=" * 50}')
    display(df.describe())

## 5. Visualisierung

### 5.1 Kursverlauf (Close-Preis in USD/Barrel)

In [ ]:
# Kursverlauf fuer alle Rohstoffe plotten
fig, axes = plt.subplots(len(oil_data), 1, figsize=(14, 4 * len(oil_data)))

if len(oil_data) == 1:
    axes = [axes]

for ax, (oil_name, df) in zip(axes, oil_data.items()):
    ax.plot(df.index, df['Close'], label=oil_name, linewidth=1)
    ax.set_title(f'Kursverlauf {oil_name} (USD/Barrel)', fontsize=14)
    ax.set_xlabel('Datum')
    ax.set_ylabel('Preis (USD)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Beide Oelpreise ueberlagert im Vergleich
fig, ax = plt.subplots(figsize=(14, 6))

for oil_name, df in oil_data.items():
    ax.plot(df.index, df['Close'], label=oil_name, linewidth=1)

ax.set_title('Oelpreisvergleich: WTI vs. Brent (USD/Barrel)', fontsize=14)
ax.set_xlabel('Datum')
ax.set_ylabel('Preis (USD)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Verteilung der taeglichen Renditen

In [ ]:
# Taegliche Renditen berechnen und Verteilung anzeigen
fig, axes = plt.subplots(1, len(oil_data), figsize=(6 * len(oil_data), 5))

if len(oil_data) == 1:
    axes = [axes]

for ax, (oil_name, df) in zip(axes, oil_data.items()):
    returns = df['Close'].pct_change().dropna()
    
    ax.hist(returns, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'Taegliche Renditen: {oil_name}', fontsize=12)
    ax.set_xlabel('Rendite (%)')
    ax.set_ylabel('Haeufigkeit')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 5.3 Fehlende Tage identifizieren (Wochenenden, Feiertage)

In [ ]:
# Luecken in den Daten identifizieren
for oil_name, df in oil_data.items():
    print(f'\n{oil_name}:')
    
    date_diffs = pd.Series(df.index).diff()
    
    # Luecken groesser als 3 Tage (Wochenende = 3 Tage normal)
    gaps = date_diffs[date_diffs > pd.Timedelta(days=3)]
    
    if len(gaps) > 0:
        print(f'  {len(gaps)} groessere Luecken gefunden (> 3 Tage):')
        for idx in gaps.index:
            print(f'    {df.index[idx-1].strftime("%Y-%m-%d")} -> {df.index[idx].strftime("%Y-%m-%d")} ({date_diffs[idx].days} Tage)')
    else:
        print('  Keine unerwarteten Luecken gefunden.')

In [ ]:
# Alle fehlenden Tage anzeigen (inkl. Wochenenden und Feiertage)
for oil_name, df in oil_data.items():
    print(f'\n{"=" * 50}')
    print(f'{oil_name}')
    print(f'{"=" * 50}')
    
    all_days = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
    missing_days = all_days.difference(df.index.normalize())
    
    missing_weekends = [d for d in missing_days if d.weekday() >= 5]
    missing_weekdays = [d for d in missing_days if d.weekday() < 5]
    
    print(f'Gesamte fehlende Tage:     {len(missing_days)}')
    print(f'  davon Wochenenden:       {len(missing_weekends)}')
    print(f'  davon Wochentage:        {len(missing_weekdays)}')
    
    if len(missing_weekdays) > 0:
        print(f'\n  Fehlende Wochentage (Feiertage etc.):')
        for d in missing_weekdays:
            day_name = ['Mo', 'Di', 'Mi', 'Do', 'Fr'][d.weekday()]
            print(f'    {d.strftime("%Y-%m-%d")} ({day_name})')

## 6. Rohdaten speichern

Die Rohdaten werden unveraendert als CSV gespeichert.

In [ ]:
# Rohdaten als CSV speichern
OUTPUT_DIR = '../../data/raw/oil/yahoo'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for oil_name, df in oil_data.items():
    filename = f'{oil_name}_{START_DATE}_to_{END_DATE}.csv'
    filepath = os.path.join(OUTPUT_DIR, filename)
    
    df.to_csv(filepath)
    print(f'Gespeichert: {filepath} ({len(df)} Zeilen)')

print('\nAlle Rohdaten gespeichert!')

## 7. Zusammenfassung

### Erkenntnisse aus der EDA:
- **Datenumfang:** (hier Ergebnisse eintragen nach Ausfuehrung)
- **Fehlende Werte:** (hier Ergebnisse eintragen)
- **Duplikate:** (hier Ergebnisse eintragen)
- **Auffaelligkeiten:** (hier Ergebnisse eintragen)

### Naechste Schritte:
1. Oelpreisdaten bereinigen und normalisieren
2. Korrelation mit Forex-Daten untersuchen
3. Oel als moeglichen Einflussfaktor auf Waehrungskurse analysieren